In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle

print("Libraries loaded successfully")

Libraries loaded successfully


In [5]:
df = pd.read_csv("/content/drive/MyDrive/FactLens_Group9/data/df_cleaned.csv")

print(f"Dataset loaded: {len(df)} articles")
print(df["label"].value_counts())


Dataset loaded: 38590 articles
label
REAL    21191
FAKE    17399
Name: count, dtype: int64


In [6]:
# X is the text - this is what goes into TF-IDF
X_text = df["cleaned_text"]

# y is the label - this is what the model will predict
y = df["label"]

print(f"Input text column shape: {X_text.shape}")
print(f"Label column shape: {y.shape}")
print(f"\nLabel values: {y.unique()}")

Input text column shape: (38590,)
Label column shape: (38590,)

Label values: ['FAKE' 'REAL']


In [7]:
# Initialize TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    max_features=50000,   # only keep the top 50,000 most important words
    ngram_range=(1, 2),   # use single words AND two-word combinations
    min_df=2,             # ignore words that appear in less than 2 articles
    max_df=0.95           # ignore words that appear in more than 95% of articles
)

# Fit and transform the cleaned text
X = vectorizer.fit_transform(X_text)

print(f"TF-IDF matrix shape: {X.shape}")
print(f"This means: {X.shape[0]} articles and {X.shape[1]} features (words)")

TF-IDF matrix shape: (38590, 50000)
This means: 38590 articles and 50000 features (words)


In [8]:
# Get the feature names (words)
feature_names = vectorizer.get_feature_names_out()

print(f"Total features (words + bigrams): {len(feature_names)}")
print(f"\nFirst 20 features:")
print(feature_names[:20])
print(f"\nLast 20 features:")
print(feature_names[-20:])

Total features (words + bigrams): 50000

First 20 features:
['aa' 'aadhaar' 'aaplo' 'aaron' 'aarp' 'ab' 'aba' 'ababa' 'aback' 'abadi'
 'abadi said' 'abandon' 'abandon nuclear' 'abandoned' 'abandoning' 'abbas'
 'abbas said' 'abbasi' 'abbott' 'abbott said']

Last 20 features:
['zinkes' 'zionist' 'zip' 'zip code' 'zipper' 'zone' 'zone budget'
 'zone finance' 'zone listen' 'zone reform' 'zone syria' 'zoning' 'zoo'
 'zucker' 'zuckerberg' 'zuckerberg said' 'zuma' 'zuma said' 'zurich'
 'zurich reuters']


In [9]:
# Convert first article to a readable format
first_article = X[0]

# Get non-zero values only (most values are zero)
non_zero = first_article.nonzero()[1]

print("Top 15 most important words in the first article:")
word_scores = [(feature_names[i], first_article[0, i]) for i in non_zero]
word_scores = sorted(word_scores, key=lambda x: x[1], reverse=True)[:15]

for word, score in word_scores:
    print(f"  {word:30} score: {score:.4f}")

Top 15 most important words in the first article:
  new year                       score: 0.4261
  december                       score: 0.3633
  hater                          score: 0.2785
  happy new                      score: 0.2409
  year                           score: 0.2094
  happy                          score: 0.1931
  year america                   score: 0.1741
  wish                           score: 0.1645
  enemy                          score: 0.1639
  new                            score: 0.1532
  december trump                 score: 0.1512
  dishonest                      score: 0.1266
  news medium                    score: 0.1220
  fake news                      score: 0.1024
  smarter                        score: 0.0998


In [10]:
# Save the TF-IDF matrix
import scipy.sparse as sp
sp.save_npz("/content/drive/MyDrive/FactLens_Group9/data/tfidf_matrix.npz", X)

# Save the vectorizer so we can use it later for new predictions
with open("/content/drive/MyDrive/FactLens_Group9/data/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

# Save the labels
y.to_csv("/content/drive/MyDrive/FactLens_Group9/data/labels.csv", index=False)

print("Saved successfully:")
print("  - tfidf_matrix.npz")
print("  - tfidf_vectorizer.pkl")
print("  - labels.csv")

Saved successfully:
  - tfidf_matrix.npz
  - tfidf_vectorizer.pkl
  - labels.csv
